In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import re, json
import torch
torch.cuda.empty_cache()

In [ ]:
# 
DATA_CSV = Path("llm/all_samples.csv")

MODEL_DIR = Path(
    "~/.cache/huggingface/"
).expanduser().resolve()

OUT_DIR = Path("llm/finetune_runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
HOLDOUT_FRAC = 0.10

MAX_LEN = 1800
BATCH_SIZE = 1
GRAD_ACCUM = 16
LR = 2e-4
EPOCHS = 3
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("MODEL_DIR exists:", MODEL_DIR.exists())
print("DATA exists:", DATA_CSV.exists())


In [ ]:
df = pd.read_csv(DATA_CSV)

required = ["context_text", "target_text", "target_speaker", "start_ms", "end_ms", "MF", "SK", "SJ"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in {DATA_CSV}: {missing}")

if "file_id" not in df.columns and "video_id" in df.columns:
    df["file_id"] = df["video_id"]
if "file_id" not in df.columns:
    raise ValueError("Need either 'file_id' or 'video_id' column in DATA_CSV.")


for l in ["MF", "SK", "SJ"]:
    df[l] = df[l].astype(int)
df["start_ms"] = df["start_ms"].astype(int)
df["end_ms"]   = df["end_ms"].astype(int)

HAS_CTX_SPEAKER = "context_speaker" in df.columns

CTX_SAME_N_WORDS = 40
CTX_DIFF_MAX_CHARS = 600

def _is_question_like(txt: str) -> bool:
    if not isinstance(txt, str):
        return False
    t = txt.strip()
    if not t:
        return False
    if "?" in t[-120:]:
        return True
    if re.match(r"^(warum|wieso|wie|was|wann|wo|welche|können|kannst|würdest|hast)\b", t.lower()):
        return True
    return False

def _tail_words(txt: str, n: int) -> str:
    if not isinstance(txt, str):
        return ""
    words = txt.strip().split()
    if len(words) <= n:
        return " ".join(words)
    return " ".join(words[-n:])

def build_model_input_ctx(row) -> str:
    ctx = str(row["context_text"]) if pd.notna(row["context_text"]) else ""
    tgt = str(row["target_text"]) if pd.notna(row["target_text"]) else ""

    if HAS_CTX_SPEAKER:
        speaker_changed = str(row["context_speaker"]) != str(row["target_speaker"])
    else:
        speaker_changed = _is_question_like(ctx)

    if speaker_changed:
        ctx_used = ctx.strip()
        if len(ctx_used) > CTX_DIFF_MAX_CHARS:
            ctx_used = ctx_used[-CTX_DIFF_MAX_CHARS:]
    else:
        ctx_used = _tail_words(ctx, CTX_SAME_N_WORDS)

    return f"[CONTEXT]\n{ctx_used}\n\n[TARGET]\n{tgt}".strip()

def build_model_input_target_only(row) -> str:
    tgt = str(row["target_text"]) if pd.notna(row["target_text"]) else ""
    return f"[TARGET]\n{tgt}".strip()

df["model_input_ctx"] = df.apply(build_model_input_ctx, axis=1)
df["model_input_tgt"] = df.apply(build_model_input_target_only, axis=1)

print("Rows:", len(df), "Videos:", df["file_id"].nunique(), "Has context_speaker:", HAS_CTX_SPEAKER)
df.head(3)


In [ ]:
from sklearn.model_selection import train_test_split

df["sig"] = df["MF"].astype(str) + df["SK"].astype(str) + df["SJ"].astype(str)

train_df, test_df = train_test_split(
    df,
    test_size=HOLDOUT_FRAC,
    random_state=SEED,
    stratify=df["sig"] if df["sig"].nunique() > 1 else None
)

print("Train:", len(train_df), "Test:", len(test_df))
print("Train label sums:\n", train_df[["MF","SK","SJ"]].sum())
print("Test label sums:\n", test_df[["MF","SK","SJ"]].sum())


In [ ]:
SYSTEM_DE = (
    "Du bist ein sehr genauer und konservativer Annotator für Selbstmitgefühl in reflexiven Interview-Transkripten von Lehrkräften in Ausbildung. "
    "Du antwortest ausschließlich im geforderten JSON-Format."
)

INSTRUCTIONS_DE = """
Klassifiziere NUR den [TARGET]-Abschnitt (nicht den Kontext) eines Interview-Transkripts
mit einer Lehrkraft in Ausbildung.

...

Antwort AUSSCHLIESSLICH als JSON:
{"MF":0,"SK":0,"SJ":0}
"""


def make_target_json(row) -> str:
    return f'{{"MF":{int(row["MF"])},"SK":{int(row["SK"])},"SJ":{int(row["SJ"])}}}'

def make_sft_example(row, variant="with_context"):
    text_blob = row["model_input_ctx"] if variant != "target_only" else row["model_input_tgt"]

    user = (
        INSTRUCTIONS_DE
        + "\n\nTEXT:\n"
        + str(text_blob).strip()
    )
    assistant = make_target_json(row)
    return user, assistant

u, a = make_sft_example(train_df.iloc[0])
print(u[:800])
print("ASSISTANT:", a)


In [ ]:
from datasets import Dataset

train_records = []
for _, row in train_df.iterrows():
    user, assistant = make_sft_example(row, variant="with_context")
    train_records.append({
        "messages": [
            {"role": "system", "content": SYSTEM_DE},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ],
        "file_id": str(row["file_id"]),
        "start_ms": int(row["start_ms"]),
        "end_ms": int(row["end_ms"]),
    })

train_ds = Dataset.from_list(train_records)


In [ ]:
def build_infer_ds(split_df, variant: str):
    records = []
    for _, row in split_df.iterrows():
        user, _assistant = make_sft_example(row, variant=variant)
        records.append({
            "messages": [
                {"role": "system", "content": SYSTEM_DE},
                {"role": "user", "content": user},
            ],
            "variant": variant,
            "file_id": str(row["file_id"]),
            "start_ms": int(row["start_ms"]),
            "end_ms": int(row["end_ms"]),
            "y_MF": int(row["MF"]), "y_SK": int(row["SK"]), "y_SJ": int(row["SJ"]),
            "target_text": str(row.get("target_text","")),
        })
    return Dataset.from_list(records)

# Holdout variants (same rows, different prompts)
test_ds_ctx = build_infer_ds(test_df, variant="with_context")
test_ds_tgt = build_infer_ds(test_df, variant="target_only")

print("train_ds:", len(train_ds))
print("test_ds_ctx:", len(test_ds_ctx))
print("test_ds_tgt:", len(test_ds_tgt))
test_ds_ctx[0]


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device_map = {"": 0} 

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    quantization_config=bnb_config,
    device_map=device_map,
    local_files_only=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = OUT_DIR / f"llama_sft_{run_id}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

model.config.use_cache = False
model.gradient_checkpointing_enable()
tokenizer.truncation_side = "left"

sft_args = SFTConfig(
    output_dir=str(RUN_DIR),
    max_length=MAX_LEN,
    packing=False,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    max_grad_norm=0.0,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    prediction_loss_only=True,
    dataloader_pin_memory=False,
    report_to=[],
)


torch.cuda.set_device(0)
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

trainer.train()

trainer.model.save_pretrained(RUN_DIR / "lora_adapter")
tokenizer.save_pretrained(RUN_DIR / "tokenizer")
print("Saved to:", RUN_DIR)


In [ ]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

OLD_RUN_DIR = Path("llm/finetune_runs/llama_sft_20260115_122520")

candidates = [
    OLD_RUN_DIR,
    OLD_RUN_DIR / "adapter",
    OLD_RUN_DIR / "final",
]
ckpts = sorted(OLD_RUN_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1)
if ckpts:
    candidates.insert(0, ckpts[-1])

ADAPTER_DIR = next((p for p in candidates if (p / "adapter_config.json").exists()), None)
assert ADAPTER_DIR is not None, f"Could not find adapter_config.json in {candidates}"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

print("Loaded BASE:", BASE_MODEL)
print("Loaded ADAPTER:", str(ADAPTER_DIR))


In [ ]:
import re, json
import numpy as np
import pandas as pd
import torch

LABELS = ["MF","SK","SJ"]
_JSON_RE_ALL = re.compile(r"\{[^{}]*\}", re.DOTALL)

def parse_json_from_text(text: str) -> dict:
    matches = _JSON_RE_ALL.findall(text or "")
    if not matches:
        return {"MF":0,"SK":0,"SJ":0}

    blob = matches[-1]
    blob = re.sub(r",\s*}", "}", blob)
    blob = re.sub(r",\s*]", "]", blob)

    try:
        obj = json.loads(blob)
    except Exception:
        return {"MF":0,"SK":0,"SJ":0}

    out = {}
    for k in LABELS:
        v = obj.get(k, 0)
        if isinstance(v, bool):
            out[k] = int(v)
        elif isinstance(v, (int, float)):
            out[k] = int(v >= 0.5)
        elif isinstance(v, str):
            out[k] = 1 if v.strip().lower() in {"1","true","ja","yes"} else 0
        else:
            out[k] = 0
    return out

N_SAMPLES = 7
GEN_KW = dict(
    max_new_tokens=40,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)

def generate_one(messages):
    model.eval()
    model.config.use_cache = True
    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass

    tokenizer.truncation_side = "left"

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, **GEN_KW)

    prompt_len = inputs["input_ids"].shape[1]
    gen_tokens = out[0, prompt_len:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

def generate_k(messages, k=N_SAMPLES):
    return [generate_one(messages) for _ in range(k)]

def run_preds(ds, variant: str):
    rows = []
    for ex in ds:
        gens = generate_k(ex["messages"], k=N_SAMPLES)
        parsed_list = [parse_json_from_text(g) for g in gens]

        prob = {l: float(np.mean([p[l] for p in parsed_list])) for l in LABELS}

        conf = {l: float(abs(prob[l] - 0.5)) for l in LABELS}

        hard = {l: int(prob[l] >= 0.5) for l in LABELS}

        rows.append({
            "variant": variant,
            "file_id": ex.get("file_id",""),
            "start_ms": int(ex.get("start_ms",-1)),
            "end_ms": int(ex.get("end_ms",-1)),
            "target_text": ex.get("target_text",""),

            "y_MF": int(ex["y_MF"]), "y_SK": int(ex["y_SK"]), "y_SJ": int(ex["y_SJ"]),

            "prob_MF": prob["MF"], "prob_SK": prob["SK"], "prob_SJ": prob["SJ"],
            "conf_MF": conf["MF"], "conf_SK": conf["SK"], "conf_SJ": conf["SJ"],

            "p_MF": hard["MF"], "p_SK": hard["SK"], "p_SJ": hard["SJ"],

            "raw_generation": gens[0],
            "raw_generation_all": "|<GEN>|".join(gens),
        })
    return pd.DataFrame(rows)

pred_ctx = run_preds(test_ds_ctx, "with_context")
pred_tgt = run_preds(test_ds_tgt, "target_only")


In [ ]:
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = OUT_DIR / f"llama_sft_{run_id}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
# save
PRED_CTX_PATH = RUN_DIR / f"preds_with_context_{run_id}.csv"
PRED_TGT_PATH = RUN_DIR / f"preds_target_only_{run_id}.csv"
pred_ctx.to_csv(PRED_CTX_PATH, index=False)
pred_tgt.to_csv(PRED_TGT_PATH, index=False)

print("Saved:", PRED_CTX_PATH)
print("Saved:", PRED_TGT_PATH)
pred_ctx.head()
